In [1]:
!pip install -U transformers accelerate sentencepiece bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 98.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 80.4 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installat

In [2]:
import os
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
)

In [39]:
# =========================
# 1. Загрузка данных
# =========================
ds = load_dataset("zjunlp/FactCHD", split="train")
ds = ds.to_pandas()

ds = ds.rename(columns={"id": "old_id"})
ds["id"] = range(len(ds))

part = 1  # номер от 1 до 5
ds_part = ds[ds["id"] % 5 == part].copy()

qa = ds_part[["id", "query", "response", "label", "reason"]].copy()

path = "/kaggle/input/datasets/stupidhouse/fact-chd-ru/factchd_ru.csv"

if os.path.exists(path):
    result_df = pd.read_csv(path)
else:
    result_df = qa.copy()
    result_df["query_ru"] = None
    result_df["response_ru"] = None

nan_mask = result_df["query_ru"].isna()

if nan_mask.any():
    start_idx = nan_mask.idxmax()
else:
    start_idx = len(result_df)

In [35]:
# =========================
# 2. Параметры
# =========================
MODEL_NAME = "google/madlad400-10b-mt"

BATCH_SIZE = 8          # для 10B лучше начать с 1
SAVE_EVERY = 1          # сохранять каждые n примеров
OUTPUT_CSV = "factchd_ru.csv"

MAX_INPUT_LENGTH = 512
MAX_NEW_TOKENS = 512

TARGET_PREFIX = "<2ru>"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

device = cuda


In [34]:
# =========================
# 3. Загрузка токенизатора и модели
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if device == "cuda":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quant_config,
        device_map="auto",
        low_cpu_mem_usage=True
    )
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    ).to(device)

model.eval()

config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/830 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.43M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.6M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

...

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/742 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(256000, 2048)
  (encoder): T5Stack(
    (embed_tokens): Embedding(256000, 2048)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear4bit(in_features=2048, out_features=4096, bias=False)
              (k): Linear4bit(in_features=2048, out_features=4096, bias=False)
              (v): Linear4bit(in_features=2048, out_features=4096, bias=False)
              (o): Linear4bit(in_features=4096, out_features=2048, bias=False)
              (relative_attention_bias): Embedding(32, 32)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear4bit(in_features=2048, out_features=16384, bias=False)
              (wi_1): Linear4bit(in_features=2048, out_feature

In [36]:
# =========================
# 4. Функция перевода батча
# =========================
@torch.inference_mode()
def translate_batch(
    texts,
    batch_size=BATCH_SIZE,
    max_input_length=MAX_INPUT_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS
):
    results = []

    texts = ["" if pd.isna(x) else str(x) for x in texts]

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        # Для MADLAD целевой язык задается префиксом
        batch = [f"{TARGET_PREFIX} {text}" for text in batch]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length
        )

        # Отправляем input на устройство модели
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        generated_tokens = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

        decoded = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True
        )
        results.extend(decoded)

        del inputs, generated_tokens
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    return results

In [ ]:
# =========================
# 5. Подготовка DataFrame
# =========================

num_rows = len(result_df)

for start in tqdm(range(start_idx, num_rows, BATCH_SIZE), desc="Translating rows"):
    end = min(start + BATCH_SIZE, num_rows)
    batch_df = result_df.iloc[start:end]

    query_ru = translate_batch(batch_df["query"].tolist(), batch_size=BATCH_SIZE)
    response_ru = translate_batch(batch_df["response"].tolist(), batch_size=BATCH_SIZE)

    result_df.loc[batch_df.index, "query_ru"] = query_ru
    result_df.loc[batch_df.index, "response_ru"] = response_ru

    processed = end
    if processed % SAVE_EVERY == 0 or processed == num_rows:
        result_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        print(f"Сохранено {processed}/{num_rows} строк в {OUTPUT_CSV}")

    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

print("Готово.")
print(result_df.head())

Translating rows:   0%|          | 0/1203 [00:00<?, ?it/s]

Сохранено 664/10277 строк в factchd_ru.csv
Сохранено 672/10277 строк в factchd_ru.csv
Сохранено 680/10277 строк в factchd_ru.csv
Сохранено 688/10277 строк в factchd_ru.csv
Сохранено 696/10277 строк в factchd_ru.csv
Сохранено 704/10277 строк в factchd_ru.csv
Сохранено 712/10277 строк в factchd_ru.csv
Сохранено 720/10277 строк в factchd_ru.csv
Сохранено 728/10277 строк в factchd_ru.csv
Сохранено 736/10277 строк в factchd_ru.csv
Сохранено 744/10277 строк в factchd_ru.csv
